In [2]:
!pip install geopy


   ---------------------------------------- 0/2 [geographiclib]
   ---------------------------------------- 0/2 [geographiclib]
   ---------------------------------------- 0/2 [geographiclib]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]



In [5]:
# import pandas as pd

# # -----------------------------
# # 1. Load datasets
# # -----------------------------
# bus_df = pd.read_excel("data/raw/ttc_bus/ttc-bus-delay-data-2024.xlsx")
# stops_df = pd.read_csv("data/raw/ttc_gtfs/stops.txt")

# # -----------------------------
# # 2. Clean text (VERY IMPORTANT)
# # -----------------------------
# bus_df['Location'] = bus_df['Location'].astype(str).str.lower().str.strip()
# stops_df['stop_name'] = stops_df['stop_name'].astype(str).str.lower().str.strip()

# # -----------------------------
# # 3. Build stop lookup set
# # -----------------------------
# stop_set = set(stops_df['stop_name'].unique())

# # -----------------------------
# # 4. Split dataset
# # -----------------------------

# # MATCHED = true GTFS stops
# matched_stops = bus_df[bus_df['Location'].isin(stop_set)].copy()

# # UNMATCHED = intersections
# intersections = bus_df[~bus_df['Location'].isin(stop_set)].copy()

# # -----------------------------
# # 5. Merge GTFS coordinates for matched stops
# # -----------------------------
# matched_stops = matched_stops.merge(
#     stops_df[['stop_name', 'stop_lat', 'stop_lon']],
#     left_on='Location',
#     right_on='stop_name',
#     how='left'
# )

# # -----------------------------
# # 6. Rename for consistency
# # -----------------------------
# matched_stops.rename(columns={
#     'stop_lat': 'lat',
#     'stop_lon': 'lon'
# }, inplace=True)

# # -----------------------------
# # 7. Geocode intersections separately (placeholder)
# # -----------------------------
# # (you will fill this after geocoding step)

# intersections['lat'] = None
# intersections['lon'] = None

# # -----------------------------
# # 8. Combine both datasets
# # -----------------------------
# final_df = pd.concat([matched_stops, intersections], ignore_index=True)

# # -----------------------------
# # 9. Save outputs for Power BI
# # -----------------------------
# final_df.to_csv("final_bus_ml_with_locations.csv", index=False)
# matched_stops.to_csv("matched_stops_only.csv", index=False)
# intersections.to_csv("intersections_only.csv", index=False)

# print("Done!")
# print("Matched stops:", len(matched_stops))
# print("Intersections:", len(intersections))

# not_matched_count = (~bus_df['Location'].isin(stop_set)).sum()
# matched_count = (bus_df['Location'].isin(stop_set)).sum()

# print("Matched:", matched_count)
# print("Not matched:", not_matched_count)

C:\Users\amras\AppData\Local\Temp\ipykernel_25048\329077211.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat([matched_stops, intersections], ignore_index=True)


Done!
Matched stops: 75260
Intersections: 48421
Matched: 11222
Not matched: 48421


In [1]:
# import pandas as pd
# from geopy.geocoders import Nominatim
# from geopy.extra.rate_limiter import RateLimiter

# # -----------------------------
# # 1. LOAD DATA
# # -----------------------------
# bus_df = pd.read_excel("data/raw/ttc_bus/ttc-bus-delay-data-2024.xlsx")
# stops_df = pd.read_csv("data/raw/ttc_gtfs/stops.txt")

# # -----------------------------
# # 2. CLEAN TEXT
# # -----------------------------
# bus_df['Location'] = bus_df['Location'].astype(str).str.lower().str.strip()
# stops_df['stop_name'] = stops_df['stop_name'].astype(str).str.lower().str.strip()

# # -----------------------------
# # 3. SPLIT MATCH / INTERSECTION
# # -----------------------------
# stop_set = set(stops_df['stop_name'].unique())

# bus_df['is_stop_match'] = bus_df['Location'].isin(stop_set)

# matched = bus_df[bus_df['is_stop_match']].copy()
# intersections = bus_df[~bus_df['is_stop_match']].copy()

# # -----------------------------
# # 4. MERGE GTFS COORDINATES (MATCHED STOPS)
# # -----------------------------
# matched = matched.merge(
#     stops_df[['stop_name', 'stop_lat', 'stop_lon']],
#     left_on='Location',
#     right_on='stop_name',
#     how='left'
# )

# matched.rename(columns={
#     'stop_lat': 'lat',
#     'stop_lon': 'lon'
# }, inplace=True)

# # -----------------------------
# # 5. SETUP GEOCODER (FOR INTERSECTIONS)
# # -----------------------------
# geolocator = Nominatim(user_agent="ttc_capstone")
# geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# # -----------------------------
# # 6. UNIQUE INTERSECTIONS ONLY
# # -----------------------------
# unique_intersections = pd.DataFrame(
#     intersections['Location'].dropna().unique(),
#     columns=['Location']
# )

# # -----------------------------
# # 7. GEOCODING FUNCTION
# # -----------------------------
# def get_coords(location):
#     try:
#         query = f"{location}, Toronto, Canada"
#         result = geocode(query)

#         if result:
#             return pd.Series([result.latitude, result.longitude])

#     except:
#         pass

#     return pd.Series([None, None])

# # -----------------------------
# # 8. APPLY GEOCODING
# # -----------------------------
# unique_intersections[['lat', 'lon']] = unique_intersections['Location'].apply(get_coords)

# # -----------------------------
# # 9. MERGE BACK TO FULL INTERSECTION DATA
# # -----------------------------
# intersections = intersections.merge(unique_intersections, on="Location", how="left")

# # -----------------------------
# # 10. COMBINE EVERYTHING
# # -----------------------------
# final_df = pd.concat([matched, intersections], ignore_index=True)

# # -----------------------------
# # 11. SAVE OUTPUTS (POWER BI READY)
# # -----------------------------
# final_df.to_csv("final_bus_geocoded.csv", index=False)
# matched.to_csv("matched_stops_geocoded.csv", index=False)
# intersections.to_csv("intersections_geocoded.csv", index=False)
# unique_intersections.to_csv("location_lookup_table.csv", index=False)

# # -----------------------------
# # 12. SUMMARY STATS
# # -----------------------------
# print("\n===== GEOCODING SUMMARY =====")
# print("Total rows:", len(bus_df))
# print("Matched GTFS stops:", len(matched))
# print("Intersection rows:", len(intersections))

# print("\nUnique locations:")
# print("GTFS matched:", matched['Location'].nunique())
# print("Intersections:", intersections['Location'].nunique())

# print("\nMissing geocodes:", intersections['lat'].isna().sum())
# print("Done successfully!")

In [2]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
from tqdm import tqdm

# -----------------------------
# 1. LOAD DATA
# -----------------------------
bus_df = pd.read_excel("data/raw/ttc_bus/ttc-bus-delay-data-2024.xlsx")
stops_df = pd.read_csv("data/raw/ttc_gtfs/stops.txt")

# -----------------------------
# 2. CLEAN TEXT
# -----------------------------
bus_df['Location'] = bus_df['Location'].astype(str).str.lower().str.strip()
stops_df['stop_name'] = stops_df['stop_name'].astype(str).str.lower().str.strip()

# -----------------------------
# 3. SPLIT MATCH / INTERSECTION
# -----------------------------
stop_set = set(stops_df['stop_name'].unique())
bus_df['is_stop_match'] = bus_df['Location'].isin(stop_set)

matched = bus_df[bus_df['is_stop_match']].copy()
intersections = bus_df[~bus_df['is_stop_match']].copy()

# -----------------------------
# 4. MERGE GTFS COORDINATES (MATCHED STOPS)
# -----------------------------
matched = matched.merge(
    stops_df[['stop_name', 'stop_lat', 'stop_lon']],
    left_on='Location',
    right_on='stop_name',
    how='left'
)
matched.rename(columns={
    'stop_lat': 'lat',
    'stop_lon': 'lon'
}, inplace=True)

# -----------------------------
# 5. SETUP GEOCODER (FOR INTERSECTIONS)
# -----------------------------
geolocator = Nominatim(user_agent="ttc_capstone")
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1,
    max_retries=2,
    error_wait_seconds=2.0,
)

# -----------------------------
# 6. UNIQUE INTERSECTIONS ONLY
# -----------------------------
unique_intersections = pd.DataFrame(
    intersections['Location'].dropna().unique(),
    columns=['Location']
)

n_unique = len(unique_intersections)
est_minutes = n_unique / 60  # ~1 request/sec
print(f"\nUnique intersections to geocode: {n_unique}")
print(f"Estimated time at ~1 req/sec: {est_minutes:.1f} minutes\n")

# -----------------------------
# 7. GEOCODING FUNCTION
# -----------------------------
def get_coords(location):
    try:
        query = f"{location}, Toronto, Ontario, Canada"
        result = geocode(query)
        if result:
            return pd.Series([result.latitude, result.longitude])
    except (GeocoderTimedOut, GeocoderServiceError) as e:
        print(f"  [geocode error] '{location}': {e}")
    except Exception as e:
        print(f"  [unexpected error] '{location}': {e}")
    return pd.Series([None, None])

# -----------------------------
# 8. APPLY GEOCODING (WITH PROGRESS BAR)
# -----------------------------
tqdm.pandas(desc="Geocoding intersections")
unique_intersections[['lat', 'lon']] = unique_intersections['Location'].progress_apply(get_coords)

# -----------------------------
# 9. MERGE BACK TO FULL INTERSECTION DATA
# -----------------------------
intersections = intersections.merge(unique_intersections, on="Location", how="left")

# -----------------------------
# 10. COMBINE EVERYTHING
# -----------------------------
final_df = pd.concat([matched, intersections], ignore_index=True)

# -----------------------------
# 11. SAVE OUTPUTS (POWER BI READY)
# -----------------------------
final_df.to_csv("final_bus_geocoded.csv", index=False)
matched.to_csv("matched_stops_geocoded.csv", index=False)
intersections.to_csv("intersections_geocoded.csv", index=False)
unique_intersections.to_csv("location_lookup_table.csv", index=False)

# -----------------------------
# 12. SUMMARY STATS
# -----------------------------
print("\n===== GEOCODING SUMMARY =====")
print("Total rows:", len(bus_df))
print("Matched GTFS stops:", len(matched))
print("Intersection rows:", len(intersections))
print("\nUnique locations:")
print("GTFS matched:", matched['Location'].nunique())
print("Intersections:", intersections['Location'].nunique())
print("\nMissing geocodes:", intersections['lat'].isna().sum(),
      f"({intersections['lat'].isna().mean():.1%} of intersection rows)")
print("Done successfully!")


Unique intersections to geocode: 10441
Estimated time at ~1 req/sec: 174.0 minutes



Geocoding intersections:   2%|▏         | 236/10441 [03:57<2:36:57,  1.08it/s]RateLimiter caught an error, retrying (0/2 tries). Called with (*('victoria park and alta, Toronto, Ontario, Canada',), **{}).
Traceback (most recent call last):
  File "C:\Users\amras\anaconda3\Lib\site-packages\urllib3\connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "C:\Users\amras\anaconda3\Lib\site-packages\urllib3\connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
  File "C:\Users\amras\anaconda3\Lib\http\client.py", line 1430, in getresponse
    response.begin()
    ~~~~~~~~~~~~~~^^
  File "C:\Users\amras\anaconda3\Lib\http\client.py", line 331, in begin
    version, status, reason = self._read_status()
                              ~~~~~~~~~~~~~~~~~^^
  File "C:\Users\amras\anaconda3\Lib\http\client.py", line 292, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ~~~~~~~~~~~~~~~~^^^^^^^^


===== GEOCODING SUMMARY =====
Total rows: 59643
Matched GTFS stops: 75260
Intersection rows: 48421

Unique locations:
GTFS matched: 64
Intersections: 10441

Missing geocodes: 40522 (83.7% of intersection rows)
Done successfully!
